In [1]:
# Cell 1 — Imports & Setup
import os
import time
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from dotenv import load_dotenv
from sqlalchemy import create_engine

# Load API keys from .env
load_dotenv('../.env')

FIRMS_KEY = os.getenv('FIRMS_MAP_KEY')
NEWS_KEY  = os.getenv('NEWS_API_KEY')

print("FIRMS key loaded:", bool(FIRMS_KEY))
print("NEWS key loaded: ", bool(NEWS_KEY))

# Database connection
engine = create_engine('postgresql://ce49x@localhost:5432/conflict_monitoring')
print("DB engine created ✓")

FIRMS key loaded: True
NEWS key loaded:  True
DB engine created ✓


In [2]:
# Cell 2 — Bölge Tanımları
REGIONS = {
    'Ukraine': {
        'bbox': '22.0,44.0,40.0,52.0',  # min_lon, min_lat, max_lon, max_lat
        'country': 'UKR'
    },
    'Yemen': {
        'bbox': '42.5,12.0,55.0,19.0',
        'country': 'YEM'
    },
    'Iraq': {
        'bbox': '38.5,29.0,48.5,37.5',
        'country': 'IRQ'
    },
    'Sudan': {
        'bbox': '21.5,8.5,38.5,22.5',
        'country': 'SDN'
    },
    'Syria': {
        'bbox': '35.5,32.5,42.5,37.5',
        'country': 'SYR'
    }
}

# Tarih aralığı: 6 ay (2024-08-01 ile 2025-02-01)
START_DATE = datetime(2024, 8, 1)
END_DATE   = datetime(2025, 2, 1)

print("Bölgeler:", list(REGIONS.keys()))
print("Tarih aralığı:", START_DATE.date(), "→", END_DATE.date())
print("Toplam gün:", (END_DATE - START_DATE).days)

Bölgeler: ['Ukraine', 'Yemen', 'Iraq', 'Sudan', 'Syria']
Tarih aralığı: 2024-08-01 → 2025-02-01
Toplam gün: 184


In [3]:
# Cell 3 — NASA FIRMS Veri Çekme Fonksiyonu
def fetch_firms_data(region_name, bbox, start_date, end_date, source='VIIRS_SNPP_SP'):
    """
    NASA FIRMS API'den 5'er günlük chunks halinde veri çeker.
    """
    all_records = []
    current_date = start_date
    
    while current_date < end_date:
        date_str = current_date.strftime('%Y-%m-%d')
        url = (
            f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
            f"{FIRMS_KEY}/{source}/{bbox}/5/{date_str}"
        )
        
        try:
            response = requests.get(url, timeout=30)
            if response.status_code == 200:
                lines = response.text.strip().split('\n')
                if len(lines) > 1:  # header + data
                    from io import StringIO
                    chunk_df = pd.read_csv(StringIO(response.text))
                    chunk_df['region'] = region_name
                    all_records.append(chunk_df)
                    print(f"  {date_str}: {len(chunk_df)} kayıt")
            else:
                print(f"  {date_str}: HTTP {response.status_code}")
        except Exception as e:
            print(f"  {date_str}: Hata — {e}")
        
        current_date += timedelta(days=5)
        time.sleep(0.5)  # API rate limit
    
    if all_records:
        return pd.concat(all_records, ignore_index=True)
    return pd.DataFrame()

print("Fonksiyon hazır ✓")

Fonksiyon hazır ✓


In [4]:
# Cell 4 — Tüm Bölgeler İçin FIRMS Verisi Çek
all_firms_dfs = []

for region_name, region_info in REGIONS.items():
    print(f"\n{'='*40}")
    print(f"Çekiliyor: {region_name}")
    print(f"{'='*40}")
    
    df = fetch_firms_data(
        region_name=region_name,
        bbox=region_info['bbox'],
        start_date=START_DATE,
        end_date=END_DATE
    )
    
    if not df.empty:
        all_firms_dfs.append(df)
        print(f"✓ {region_name}: {len(df)} toplam kayıt")
    else:
        print(f"✗ {region_name}: veri gelmedi")

# Tüm bölgeleri birleştir
df_firms_raw = pd.concat(all_firms_dfs, ignore_index=True)
print(f"\n{'='*40}")
print(f"TOPLAM KAYIT: {len(df_firms_raw)}")
print(f"{'='*40}")


Çekiliyor: Ukraine
  2024-08-01: 2472 kayıt
  2024-08-06: 1852 kayıt
  2024-08-11: 3337 kayıt
  2024-08-16: 4998 kayıt
  2024-08-21: 5015 kayıt
  2024-08-26: 5066 kayıt
  2024-08-31: 6588 kayıt
  2024-09-05: 6373 kayıt
  2024-09-10: 2832 kayıt
  2024-09-15: 8841 kayıt
  2024-09-20: 4510 kayıt
  2024-09-25: 3795 kayıt
  2024-09-30: 5935 kayıt
  2024-10-05: 2604 kayıt
  2024-10-10: 955 kayıt
  2024-10-15: 262 kayıt
  2024-10-20: 748 kayıt
  2024-10-25: 657 kayıt
  2024-10-30: 386 kayıt
  2024-11-04: 402 kayıt
  2024-11-09: 85 kayıt
  2024-11-14: 170 kayıt
  2024-11-19: 179 kayıt
  2024-11-24: 79 kayıt
  2024-11-29: 30 kayıt
  2024-12-04: 29 kayıt
  2024-12-09: 40 kayıt
  2024-12-14: 123 kayıt
  2024-12-19: 84 kayıt
  2024-12-24: 22 kayıt
  2024-12-29: 112 kayıt
  2025-01-03: 148 kayıt
  2025-01-08: 178 kayıt
  2025-01-13: 106 kayıt
  2025-01-18: 140 kayıt
  2025-01-23: 101 kayıt
  2025-01-28: 208 kayıt
✓ Ukraine: 69462 toplam kayıt

Çekiliyor: Yemen
  2024-08-01: 12 kayıt
  2024-08-06: 

In [5]:
# Cell 5 — Veri Temizleme
print("Ham veri shape:", df_firms_raw.shape)
print("\nSütunlar:", df_firms_raw.columns.tolist())
print("\nEksik değerler:\n", df_firms_raw.isnull().sum())

Ham veri shape: (622922, 16)

Sütunlar: ['latitude', 'longitude', 'bright_ti4', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_ti5', 'frp', 'daynight', 'type', 'region']

Eksik değerler:
 latitude      0
longitude     0
bright_ti4    0
scan          0
track         0
acq_date      0
acq_time      0
satellite     0
instrument    0
confidence    0
version       0
bright_ti5    0
frp           0
daynight      0
type          0
region        0
dtype: int64


In [6]:
# Cell 6 — Veri Temizleme & Hazırlama
df_firms = df_firms_raw.copy()

# 1. acq_date'i datetime'a çevir
df_firms['acq_date'] = pd.to_datetime(df_firms['acq_date'])

# 2. Confidence filtresi — VIIRS'te confidence: 'low', 'nominal', 'high'
#    Sadece 'nominal' ve 'high' tutuyoruz (low = gürültülü detections)
print("Confidence dağılımı (ham):")
print(df_firms['confidence'].value_counts())

before = len(df_firms)
df_firms = df_firms[df_firms['confidence'].isin(['nominal', 'high', 'n', 'h'])]
after = len(df_firms)
print(f"\nLow-confidence filtrelemesi: {before} → {after} ({before-after} kayıt çıkarıldı)")

# 3. FRP anomaly temizliği (negatif veya aşırı yüksek değerler)
df_firms = df_firms[df_firms['frp'] >= 0]
df_firms = df_firms[df_firms['frp'] < 10000]  # 10000 MW üzeri fiziksel olarak imkansız

# 4. brightness için de temel kontrol
df_firms = df_firms[df_firms['bright_ti4'] > 200]  # Kelvin — 200K altı geçersiz

# 5. Duplicate temizliği
df_firms = df_firms.drop_duplicates(subset=['latitude','longitude','acq_date','acq_time'])

print(f"\nTemizlik sonrası toplam kayıt: {len(df_firms)}")
print("\nBölge dağılımı:")
print(df_firms['region'].value_counts())

Confidence dağılımı (ham):
confidence
n    505580
l     80455
h     36887
Name: count, dtype: int64

Low-confidence filtrelemesi: 622922 → 542467 (80455 kayıt çıkarıldı)

Temizlik sonrası toplam kayıt: 533260

Bölge dağılımı:
region
Sudan      349513
Iraq       110367
Ukraine     65577
Syria        5192
Yemen        2611
Name: count, dtype: int64


In [7]:
# Cell 7 — Keşifsel İnceleme & PostgreSQL'e Yaz
print("=== df.head() ===")
display(df_firms.head())

print("\n=== df.describe() ===")
display(df_firms.describe())

print("\n=== df.info() ===")
df_firms.info()

=== df.head() ===


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type,region
0,47.83243,38.48219,303.33,0.70,0.75,2024-08-01,52,N,VIIRS,n,2,285.34,2.67,N,2,Ukraine
1,47.83516,38.47837,304.22,0.70,0.75,2024-08-01,52,N,VIIRS,n,2,284.94,1.65,N,2,Ukraine
2,48.26630,37.92342,296.40,0.65,0.73,2024-08-01,52,N,VIIRS,n,2,282.29,1.14,N,0,Ukraine
3,48.39121,37.91235,296.42,0.64,0.72,2024-08-01,52,N,VIIRS,n,2,283.59,1.12,N,0,Ukraine
4,48.45673,38.76737,297.05,0.68,0.74,2024-08-01,52,N,VIIRS,n,2,284.21,2.02,N,0,Ukraine



=== df.describe() ===


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,version,bright_ti5,frp,type
count,533260.000000,533260.000000,533260.000000,533260.000000,533260.000000,533260,533260.000000,533260.0,533260.000000,533260.000000,533260.000000
mean,20.071603,33.606627,333.737199,0.466890,0.497007,2024-11-23 08:33:10.551700,1388.647733,2.0,299.259545,8.053179,0.342178
min,8.500000,21.500030,295.000000,0.320000,0.360000,2024-08-01 00:00:00,0.000000,2.0,236.790000,0.000000,0.000000
25%,9.726875,26.505635,317.880000,0.400000,0.390000,2024-10-22 00:00:00,1048.000000,2.0,290.960000,2.210000,0.000000
50%,11.265000,33.131005,338.070000,0.440000,0.470000,2024-12-05 00:00:00,1132.000000,2.0,300.960000,4.830000,0.000000
75%,31.762230,37.868383,346.330000,0.520000,0.590000,2024-12-31 00:00:00,2223.000000,2.0,307.070000,9.050000,0.000000
max,51.999930,54.648080,367.000000,0.800000,0.780000,2025-02-01 00:00:00,2359.000000,2.0,371.960000,552.250000,3.000000
std,14.175068,7.938664,19.065848,0.093275,0.120379,NaN,643.089666,0.0,10.595101,12.305068,0.760467



=== df.info() ===
<class 'pandas.DataFrame'>
Index: 533260 entries, 0 to 622921
Data columns (total 16 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   latitude    533260 non-null  float64       
 1   longitude   533260 non-null  float64       
 2   bright_ti4  533260 non-null  float64       
 3   scan        533260 non-null  float64       
 4   track       533260 non-null  float64       
 5   acq_date    533260 non-null  datetime64[us]
 6   acq_time    533260 non-null  int64         
 7   satellite   533260 non-null  str           
 8   instrument  533260 non-null  str           
 9   confidence  533260 non-null  str           
 10  version     533260 non-null  int64         
 11  bright_ti5  533260 non-null  float64       
 12  frp         533260 non-null  float64       
 13  daynight    533260 non-null  str           
 14  type        533260 non-null  int64         
 15  region      533260 non-null  str           
dtyp

In [8]:
# Cell 8 — PostgreSQL'e Yaz
print("firms_detections tablosuna yazılıyor...")

df_firms.to_sql(
    'firms_detections',
    engine,
    if_exists='replace',
    index=False
)

print(f"✓ {len(df_firms)} kayıt yazıldı!")

# Doğrulama — geri oku
df_check = pd.read_sql("SELECT COUNT(*) as total, region FROM firms_detections GROUP BY region ORDER BY total DESC", engine)
print("\nVeritabanından doğrulama:")
display(df_check)

firms_detections tablosuna yazılıyor...
✓ 533260 kayıt yazıldı!

Veritabanından doğrulama:


,total,region
0,349513,Sudan
1,110367,Iraq
2,65577,Ukraine
3,5192,Syria
4,2611,Yemen


In [9]:
# Cell 9 — NewsAPI ile Haber Toplama
import time

def fetch_news(query, from_date, to_date, api_key, page_size=100):
    """NewsAPI'den haber çeker."""
    all_articles = []
    page = 1
    
    while True:
        url = "https://newsapi.org/v2/everything"
        params = {
            'q': query,
            'from': from_date,
            'to': to_date,
            'language': 'en',
            'sortBy': 'publishedAt',
            'pageSize': page_size,
            'page': page,
            'apiKey': api_key
        }
        
        response = requests.get(url, params=params)
        data = response.json()
        
        if data.get('status') != 'ok':
            print(f"  Hata: {data.get('message', 'bilinmeyen')}")
            break
            
        articles = data.get('articles', [])
        if not articles:
            break
            
        all_articles.extend(articles)
        total = data.get('totalResults', 0)
        print(f"  Sayfa {page}: {len(articles)} makale (toplam: {total})")
        
        if len(all_articles) >= min(total, 1000):
            break
        
        page += 1
        time.sleep(0.5)
    
    return all_articles

print("Fonksiyon hazır ✓")

Fonksiyon hazır ✓


In [10]:
# Cell 10 — Haberleri Çek
CONFLICT_QUERIES = [
    'Ukraine war military attack',
    'Yemen conflict Houthi attack',
    'Iraq war bombing military',
    'Sudan conflict war fighting',
    'Syria war conflict airstrike'
]

# NewsAPI ücretsiz plan: son 1 ay — biz arşiv için farklı sorgular yapacağız
FROM_DATE = '2024-08-01'
TO_DATE   = '2025-01-31'

all_articles_raw = []

for query in CONFLICT_QUERIES:
    print(f"\n--- Sorgu: '{query}' ---")
    articles = fetch_news(
        query=query,
        from_date=FROM_DATE,
        to_date=TO_DATE,
        api_key=NEWS_KEY
    )
    print(f"  → {len(articles)} makale toplandı")
    for a in articles:
        a['search_query'] = query
    all_articles_raw.extend(articles)

print(f"\nTOPLAM HAM MAKALE: {len(all_articles_raw)}")


--- Sorgu: 'Ukraine war military attack' ---
  Hata: You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2026-04-25, but you have requested 2024-08-01. You may need to upgrade to a paid plan.
  → 0 makale toplandı

--- Sorgu: 'Yemen conflict Houthi attack' ---
  Hata: You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2026-04-25, but you have requested 2024-08-01. You may need to upgrade to a paid plan.
  → 0 makale toplandı

--- Sorgu: 'Iraq war bombing military' ---
  Hata: You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2026-04-25, but you have requested 2024-08-01. You may need to upgrade to a paid plan.
  → 0 makale toplandı

--- Sorgu: 'Sudan conflict war fighting' ---
  Hata: You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2026-04-25

In [11]:
# Cell 10 (güncellendi) — Son 1 Ay Haberleri
FROM_DATE = '2026-04-25'
TO_DATE   = '2026-05-26'

all_articles_raw = []

for query in CONFLICT_QUERIES:
    print(f"\n--- Sorgu: '{query}' ---")
    articles = fetch_news(
        query=query,
        from_date=FROM_DATE,
        to_date=TO_DATE,
        api_key=NEWS_KEY
    )
    print(f"  → {len(articles)} makale toplandı")
    for a in articles:
        a['search_query'] = query
    all_articles_raw.extend(articles)

print(f"\nTOPLAM HAM MAKALE: {len(all_articles_raw)}")


--- Sorgu: 'Ukraine war military attack' ---
  Hata: You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2026-04-25, but you have requested 2026-04-25. You may need to upgrade to a paid plan.
  → 0 makale toplandı

--- Sorgu: 'Yemen conflict Houthi attack' ---
  Hata: You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2026-04-25, but you have requested 2026-04-25. You may need to upgrade to a paid plan.
  → 0 makale toplandı

--- Sorgu: 'Iraq war bombing military' ---
  Hata: You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2026-04-25, but you have requested 2026-04-25. You may need to upgrade to a paid plan.
  → 0 makale toplandı

--- Sorgu: 'Sudan conflict war fighting' ---
  Hata: You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2026-04-25

In [12]:
load_dotenv('../.env', override=True)
GNEWS_KEY = os.getenv('GNEWS_API_KEY')
print("GNews key loaded:", bool(GNEWS_KEY))

GNews key loaded: True


In [13]:
# Cell 12 — GNews ile Haber Çekme
def fetch_gnews(query, api_key, lang='en', max_results=100):
    """GNews API'den haber çeker."""
    all_articles = []
    
    url = "https://gnews.io/api/v4/search"
    params = {
        'q': query,
        'lang': lang,
        'max': 10,  # ücretsiz planda max 10
        'apikey': api_key
    }
    
    try:
        response = requests.get(url, params=params, timeout=15)
        data = response.json()
        
        if 'articles' in data:
            articles = data['articles']
            for a in articles:
                a['search_query'] = query
            all_articles.extend(articles)
            print(f"  → {len(articles)} makale (toplam mevcut: {data.get('totalArticles', '?')})")
        else:
            print(f"  Hata: {data}")
    except Exception as e:
        print(f"  Exception: {e}")
    
    return all_articles

# Çatışma sorguları
QUERIES = [
    'Ukraine war attack',
    'Yemen Houthi conflict',
    'Iraq military bombing',
    'Sudan war fighting',
    'Syria airstrike conflict',
    'Ukraine missile shelling',
    'Yemen Red Sea attack',
    'Iraq explosion troops',
    'Sudan armed conflict',
    'Syria rebel offensive'
]

all_articles_raw = []

for query in QUERIES:
    print(f"\nSorgu: '{query}'")
    articles = fetch_gnews(query, GNEWS_KEY)
    all_articles_raw.extend(articles)
    time.sleep(1)

print(f"\nTOPLAM HAM MAKALE: {len(all_articles_raw)}")


Sorgu: 'Ukraine war attack'
  → 10 makale (toplam mevcut: 3902)

Sorgu: 'Yemen Houthi conflict'
  → 6 makale (toplam mevcut: 289)

Sorgu: 'Iraq military bombing'
  → 0 makale (toplam mevcut: 8)

Sorgu: 'Sudan war fighting'
  → 6 makale (toplam mevcut: 261)

Sorgu: 'Syria airstrike conflict'
  → 0 makale (toplam mevcut: 6)

Sorgu: 'Ukraine missile shelling'
  → 1 makale (toplam mevcut: 117)

Sorgu: 'Yemen Red Sea attack'
  → 2 makale (toplam mevcut: 490)

Sorgu: 'Iraq explosion troops'
  → 0 makale (toplam mevcut: 2)

Sorgu: 'Sudan armed conflict'
  → 4 makale (toplam mevcut: 69)

Sorgu: 'Syria rebel offensive'
  → 0 makale (toplam mevcut: 119)

TOPLAM HAM MAKALE: 29


In [14]:
print("GNEWS_KEY değeri:", GNEWS_KEY)

GNEWS_KEY değeri: 87c80b9457a3a0dda32183f1f532af15


In [15]:
# Key'i yeniden yükle
load_dotenv('../.env', override=True)
GNEWS_KEY = os.getenv('GNEWS_API_KEY')
print("GNews key:", GNEWS_KEY)

GNews key: 87c80b9457a3a0dda32183f1f532af15


In [16]:
# Cell 16 — GNews Haber Çekme (key güncellendi)
all_articles_raw = []

for query in QUERIES:
    print(f"\nSorgu: '{query}'")
    articles = fetch_gnews(query, GNEWS_KEY)
    all_articles_raw.extend(articles)
    time.sleep(1)

print(f"\nTOPLAM HAM MAKALE: {len(all_articles_raw)}")


Sorgu: 'Ukraine war attack'
  → 10 makale (toplam mevcut: 3902)

Sorgu: 'Yemen Houthi conflict'
  → 6 makale (toplam mevcut: 289)

Sorgu: 'Iraq military bombing'
  → 0 makale (toplam mevcut: 8)

Sorgu: 'Sudan war fighting'
  → 6 makale (toplam mevcut: 261)

Sorgu: 'Syria airstrike conflict'
  → 0 makale (toplam mevcut: 6)

Sorgu: 'Ukraine missile shelling'
  → 1 makale (toplam mevcut: 117)

Sorgu: 'Yemen Red Sea attack'
  → 2 makale (toplam mevcut: 490)

Sorgu: 'Iraq explosion troops'
  → 0 makale (toplam mevcut: 2)

Sorgu: 'Sudan armed conflict'
  → 4 makale (toplam mevcut: 69)

Sorgu: 'Syria rebel offensive'
  → 0 makale (toplam mevcut: 119)

TOPLAM HAM MAKALE: 29


In [17]:
# Cell 17 — Daha Fazla Sorgu ile 1000+ Makale
EXTRA_QUERIES = [
    'war conflict 2025',
    'military strike bombing 2025',
    'armed conflict troops',
    'missile attack explosion',
    'combat shelling warfare',
    'Ukraine Russia war',
    'Middle East conflict',
    'Africa war Sudan',
    'ceasefire violation attack',
    'war casualties military',
    'airstrike bombing civilians',
    'rebel offensive troops',
    'naval attack Red Sea',
    'drone strike military',
    'war crimes conflict zone',
    'peace talks war breakdown',
    'military offensive advance',
    'conflict escalation 2025',
    'war displacement refugees',
    'armed forces attack region'
]

for query in EXTRA_QUERIES:
    print(f"Sorgu: '{query}'")
    articles = fetch_gnews(query, GNEWS_KEY)
    all_articles_raw.extend(articles)
    time.sleep(1.2)

print(f"\nTOPLAM HAM MAKALE: {len(all_articles_raw)}")

Sorgu: 'war conflict 2025'
  → 10 makale (toplam mevcut: 143)
Sorgu: 'military strike bombing 2025'
  → 0 makale (toplam mevcut: 0)
Sorgu: 'armed conflict troops'
  → 3 makale (toplam mevcut: 53)
Sorgu: 'missile attack explosion'
  → 1 makale (toplam mevcut: 38)
Sorgu: 'combat shelling warfare'
  → 0 makale (toplam mevcut: 0)
Sorgu: 'Ukraine Russia war'
  → 10 makale (toplam mevcut: 46832)
Sorgu: 'Middle East conflict'
  → 10 makale (toplam mevcut: 15617)
Sorgu: 'Africa war Sudan'
  → 3 makale (toplam mevcut: 71)
Sorgu: 'ceasefire violation attack'
  → 0 makale (toplam mevcut: 84)
Sorgu: 'war casualties military'
  → 10 makale (toplam mevcut: 314)
Sorgu: 'airstrike bombing civilians'
  → 1 makale (toplam mevcut: 10)
Sorgu: 'rebel offensive troops'
  → 0 makale (toplam mevcut: 5)
Sorgu: 'naval attack Red Sea'
  → 0 makale (toplam mevcut: 30)
Sorgu: 'drone strike military'
  → 10 makale (toplam mevcut: 813)
Sorgu: 'war crimes conflict zone'
  → 0 makale (toplam mevcut: 3)
Sorgu: 'peace t

In [18]:
# Mevcut haberleri geçici olarak kaydet
import json
with open('../data_backup.json', 'w') as f:
    json.dump(all_articles_raw, f)
print(f"✓ {len(all_articles_raw)} makale yedeklendi")

✓ 97 makale yedeklendi


In [19]:
load_dotenv('../.env', override=True)
GUARDIAN_KEY = os.getenv('GUARDIAN_API_KEY')
print("Guardian key:", GUARDIAN_KEY)

Guardian key: 237fddc9-8cf3-4f85-98cf-c1d6b2f9f70a


In [20]:
# Cell 20 — Guardian API ile Haber Çekme
def fetch_guardian(query, api_key, from_date='2024-08-01', to_date='2025-02-01', pages=10):
    """Guardian API'den haber çeker — ücretsiz, geniş arşiv."""
    all_articles = []
    
    for page in range(1, pages + 1):
        url = "https://content.guardianapis.com/search"
        params = {
            'q': query,
            'from-date': from_date,
            'to-date': to_date,
            'page': page,
            'page-size': 50,
            'show-fields': 'headline,bodyText,shortUrl',
            'api-key': api_key
        }
        
        try:
            response = requests.get(url, params=params, timeout=15)
            data = response.json()
            
            results = data.get('response', {}).get('results', [])
            if not results:
                break
                
            for r in results:
                all_articles.append({
                    'title': r.get('webTitle', ''),
                    'publishedAt': r.get('webPublicationDate', ''),
                    'source': 'The Guardian',
                    'url': r.get('webUrl', ''),
                    'search_query': query,
                    'description': r.get('fields', {}).get('bodyText', '')[:300] if r.get('fields') else ''
                })
            
            total_pages = data['response'].get('pages', 1)
            if page >= total_pages:
                break
                
            time.sleep(0.3)
            
        except Exception as e:
            print(f"  Hata: {e}")
            break
    
    return all_articles

# Sorgular
GUARDIAN_QUERIES = [
    'Ukraine war', 'Ukraine Russia military',
    'Yemen conflict Houthi', 'Yemen war',
    'Iraq war bombing', 'Iraq military',
    'Sudan conflict war', 'Sudan fighting',
    'Syria war airstrike', 'Syria conflict'
]

guardian_articles = []

for query in GUARDIAN_QUERIES:
    print(f"Sorgu: '{query}'")
    articles = fetch_guardian(query, GUARDIAN_KEY)
    guardian_articles.extend(articles)
    print(f"  → {len(articles)} makale")
    time.sleep(0.5)

print(f"\nGuardian toplam: {len(guardian_articles)}")

Sorgu: 'Ukraine war'
  → 500 makale
Sorgu: 'Ukraine Russia military'
  → 500 makale
Sorgu: 'Yemen conflict Houthi'
  → 500 makale
Sorgu: 'Yemen war'
  → 500 makale
Sorgu: 'Iraq war bombing'
  → 500 makale
Sorgu: 'Iraq military'
  → 500 makale
Sorgu: 'Sudan conflict war'
  → 500 makale
Sorgu: 'Sudan fighting'
  → 500 makale
Sorgu: 'Syria war airstrike'
  → 500 makale
Sorgu: 'Syria conflict'
  → 500 makale

Guardian toplam: 5000


In [21]:
# Cell 21 — Haberleri Birleştir & DataFrame Yap
import json

# GNews makalelerini Guardian formatına dönüştür
gnews_formatted = []
for a in all_articles_raw:
    gnews_formatted.append({
        'title': a.get('title', ''),
        'publishedAt': a.get('publishedAt', ''),
        'source': a.get('source', {}).get('name', 'GNews') if isinstance(a.get('source'), dict) else 'GNews',
        'url': a.get('url', ''),
        'search_query': a.get('search_query', ''),
        'description': a.get('description', '') or ''
    })

# Birleştir
all_formatted = gnews_formatted + guardian_articles
df_news_raw = pd.DataFrame(all_formatted)

print(f"Toplam ham makale: {len(df_news_raw)}")
print(f"\nKaynak dağılımı:")
print(df_news_raw['source'].value_counts().head(10))

# Duplicate temizle
df_news = df_news_raw.drop_duplicates(subset=['title'])
df_news = df_news[df_news['title'].str.len() > 5]
df_news['publishedAt'] = pd.to_datetime(df_news['publishedAt'], errors='coerce')

# Bölge etiketi ekle
def assign_region(row):
    text = (str(row['title']) + ' ' + str(row['search_query'])).lower()
    if any(w in text for w in ['ukraine', 'kyiv', 'zelensky', 'russia']):
        return 'Ukraine'
    elif any(w in text for w in ['yemen', 'houthi', 'sanaa']):
        return 'Yemen'
    elif any(w in text for w in ['iraq', 'baghdad', 'mosul']):
        return 'Iraq'
    elif any(w in text for w in ['sudan', 'khartoum', 'darfur']):
        return 'Sudan'
    elif any(w in text for w in ['syria', 'damascus', 'aleppo']):
        return 'Syria'
    else:
        return 'General'

df_news['region'] = df_news.apply(assign_region, axis=1)

print(f"\nTemizlik sonrası: {len(df_news)} makale")
print(f"\nBölge dağılımı:")
print(df_news['region'].value_counts())
display(df_news.head())

Toplam ham makale: 5097

Kaynak dağılımı:
source
The Guardian                        5002
Devdiscourse                          19
Times Now                              9
The Atlanta Journal-Constitution       6
Daily Excelsior                        4
The Straits Times                      4
Al-Monitor                             4
The Economic Times                     4
The Morning Star                       3
Breitbart News Network                 3
Name: count, dtype: int64

Temizlik sonrası: 2372 makale

Bölge dağılımı:
region
Ukraine    680
Yemen      628
Sudan      419
Iraq       373
Syria      234
General     38
Name: count, dtype: int64


,title,publishedAt,source,url,search_query,description,region
0,Leaders keep a wary eye on Belarus for signs i...,2026-05-25 19:41:19+00:00,The Atlanta Journal-Constitution,https://www.ajc.com/news/2026/05/leaders-keep-...,Ukraine war attack,Belarus' exiled opposition leader visits Kyiv ...,Ukraine
1,Ukraine-Russia war latest: Zelensky demands ac...,2026-05-25 12:00:34+00:00,The Independent,https://www.independent.co.uk/news/world/europ...,Ukraine war attack,At least four people were killed in the attack...,Ukraine
2,"Russian drones, hypersonic missiles batter Kyi...",2026-05-25 07:55:00+00:00,Arkansas Online,https://www.arkansasonline.com/news/2026/may/2...,Ukraine war attack,"KYIV, Ukraine -- Russia used the powerful hype...",Ukraine
3,Ukraine war briefing: Putin accused of ‘reckle...,2026-05-25 01:06:42+00:00,The Guardian,https://www.theguardian.com/world/2026/may/25/...,Ukraine war attack,Russia’s deadly attack condemned across Europe...,Ukraine
4,Russia attacks Ukraine using the hypersonic Or...,2026-05-25 00:00:00+00:00,The Morning Star,https://morningstaronline.co.uk/article/russia...,Ukraine war attack,RUSSIA used its powerful hypersonic Oreshnik b...,Ukraine


In [22]:
# Cell 22 — Haberleri PostgreSQL'e Yaz
print("news_articles tablosuna yazılıyor...")

df_news.to_sql(
    'news_articles',
    engine,
    if_exists='replace',
    index=False
)

print(f"✓ {len(df_news)} makale yazıldı!")

# Doğrulama
df_check_news = pd.read_sql("""
    SELECT region, COUNT(*) as total, COUNT(DISTINCT source) as sources
    FROM news_articles 
    GROUP BY region 
    ORDER BY total DESC
""", engine)

print("\nVeritabanından doğrulama:")
display(df_check_news)

news_articles tablosuna yazılıyor...
✓ 2372 makale yazıldı!

Veritabanından doğrulama:


,region,total,sources
0,Ukraine,680,18
1,Yemen,628,5
2,Sudan,419,8
3,Iraq,373,1
4,Syria,234,1
5,General,38,20
